In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

In [ ]:
# Load results from trading_daily notebook
results_file = MODEL_DATA_DIR / 'trading_daily_results.pkl'
with open(results_file, 'rb') as f:
    results = pickle.load(f)

portfolios = results['portfolios']
portfolios_filtered = results['portfolios_filtered']
ff_factors = results['ff_factors']
prediction_columns = results['prediction_columns']
MODELS = results['MODELS']

print(f"Loaded results with {len(prediction_columns)} models")
print(f"Models: {list(MODELS.keys())}")

# Plot log cumulative returns

In [ ]:
# Group models for plotting
model_groups = {
    'Two Features': ['lr', 'lasso', 'elasticnet', 'nn', 'nn_tuned_1layer', 'nn_tuned_2layer', 'nn_tuned_3layer', 'nn_tuned_4layer'],
    'All Features': ['lr_all', 'lasso_all', 'elasticnet_all', 'nn_all', 'nn_tuned_1layer_all', 'nn_tuned_2layer_all', 'nn_tuned_3layer_all', 'nn_tuned_4layer_all']
}

def safe_filename(name):
    """Create a filename-safe version of model names"""
    return name.lower().replace(' ', '_').replace('(', '').replace(')', '')

for group_name, model_keys in model_groups.items():
    # Get the prediction columns that exist
    pred_cols = [MODELS[k]['col'] for k in model_keys if k in MODELS and MODELS[k]['col'] in portfolios]
    
    if not pred_cols:
        print(f"Skipping {group_name} - no portfolios available")
        continue
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for pred_col in pred_cols:
        portfolio = portfolios[pred_col]
        
        # Calculate long-short returns
        ls_ret = portfolio['long_ret'].fillna(0) - portfolio['short_ret'].fillna(0)
        
        # Calculate log cumulative returns
        log_cum_ret = np.log(1 + ls_ret).cumsum()
        
        # Get model name for legend
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        
        # Plot
        ax.plot(log_cum_ret.index, log_cum_ret.values, label=model_name, linewidth=1.5)
    
    ax.set_title(f'Log Cumulative Returns - {group_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Log Cumulative Return', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    
    plt.tight_layout()
    
    # Save the figure
    filename = f'log_cumulative_returns_{safe_filename(group_name)}.png'
    filepath = FIGURES_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"Saved: {filename}")
    
    plt.show()

# Plot log cumulative returns: Long and Short legs

In [ ]:
# Plot long and short legs separately across all models
pred_cols_all = [MODELS[k]['col'] for k in MODELS if MODELS[k]['col'] in portfolios]

for leg, ret_col in [('Long', 'long_ret'), ('Short', 'short_ret')]:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for pred_col in pred_cols_all:
        portfolio = portfolios[pred_col]
        ret = portfolio[ret_col].fillna(0)
        log_cum_ret = np.log(1 + ret).cumsum()
        
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        ax.plot(log_cum_ret.index, log_cum_ret.values, label=model_name, linewidth=1.5)
    
    ax.set_title(f'Log Cumulative Returns - {leg} Portfolio', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Log Cumulative Return', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    
    plt.tight_layout()
    
    filename = f'log_cumulative_returns_{leg.lower()}_portfolio.png'
    filepath = FIGURES_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"Saved: {filename}")
    
    plt.show()

# Number of stocks in Long and Short portfolios

In [ ]:
# Plot number of stocks in long and short legs over time
pred_cols_all = [MODELS[k]['col'] for k in MODELS if MODELS[k]['col'] in portfolios]

for leg, n_col in [('Long', 'n_long'), ('Short', 'n_short')]:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for pred_col in pred_cols_all:
        portfolio = portfolios[pred_col]
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        ax.plot(portfolio.index, portfolio[n_col], label=model_name, linewidth=1.5)
    
    ax.set_title(f'Number of Stocks - {leg} Portfolio', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('Number of Stocks', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    filename = f'n_stocks_{leg.lower()}_portfolio.png'
    filepath = FIGURES_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"Saved: {filename}")
    
    plt.show()